In [1]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import pennylane as qml

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cpu


In [2]:
SEED = 42
BATCH_SIZE = 64
EPOCHS = 5
LR = 1e-3
N_QUBITS = 4
N_Q_LAYERS = 2
PATCH_SIZE = 7
EMBED_DIM = 16
torch.manual_seed(SEED)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

train_ds = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_ds = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

In [6]:
class QuantumLayer(nn.Module):
    def __init__(self, n_qubits=4, n_layers=2):
        super().__init__()
        dev = qml.device('default.qubit', wires=n_qubits)

        @qml.qnode(dev, interface='torch')
        def circuit(inputs, weights):
            for i in range(n_qubits):
                qml.RY(inputs[i], wires=i)
            for l in range(n_layers):
                for i in range(n_qubits):
                    qml.RX(weights[l][i], wires=i)
                    qml.RZ(weights[l][i], wires=i)
                for i in range(n_qubits - 1):
                    qml.CNOT(wires=[i, i + 1])
            return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

        self.layer = qml.qnn.TorchLayer(circuit, {'weights': (n_layers, n_qubits)})

    def forward(self, x):
        orig_device = x.device
        q_in = x.cpu() if x.is_cuda else x
        if q_in.dim() == 1:
            out = self.layer(q_in)
            return out.to(orig_device) if orig_device.type == 'cuda' else out
        # TorchLayer expects single-sample input for this circuit; map across batch.
        out = torch.stack([self.layer(sample) for sample in q_in], dim=0)
        return out.to(orig_device) if orig_device.type == 'cuda' else out

class PatchEmbedding(nn.Module):
    def __init__(self, patch_size=7, embed_dim=16):
        super().__init__()
        self.proj = nn.Conv2d(1, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x)
        x = x.flatten(2).transpose(1, 2)
        return x

class HQViT(nn.Module):
    def __init__(self, patch_size=7, embed_dim=16, n_qubits=4, n_q_layers=2, num_classes=10):
        super().__init__()
        self.patch_embed = PatchEmbedding(patch_size=patch_size, embed_dim=embed_dim)
        self.fc_reduce = nn.Linear(embed_dim, n_qubits)
        self.quantum = QuantumLayer(n_qubits=n_qubits, n_layers=n_q_layers)
        self.classifier = nn.Linear(n_qubits, num_classes)

    def forward(self, x):
        x = self.patch_embed(x)
        x = x.mean(dim=1)
        x = self.fc_reduce(x)
        x = self.quantum(x)
        return self.classifier(x)

model = HQViT(patch_size=PATCH_SIZE, embed_dim=EMBED_DIM, n_qubits=N_QUBITS, n_q_layers=N_Q_LAYERS).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

In [7]:
def train_model(model, train_loader, test_loader, epochs):
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    if device.type == 'cuda':
        torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()
    for epoch in range(epochs):
        model.train()
        running_loss, n = 0.0, 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * x.size(0)
            n += x.size(0)
        train_loss = running_loss / n

        model.eval()
        vloss, correct, m = 0.0, 0, 0
        with torch.no_grad():
            for x, y in test_loader:
                x, y = x.to(device), y.to(device)
                logits = model(x)
                loss = criterion(logits, y)
                vloss += loss.item() * x.size(0)
                correct += (logits.argmax(dim=1) == y).sum().item()
                m += x.size(0)
        val_loss = vloss / m
        val_acc = correct / m
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        print(f'Epoch {epoch+1}/{epochs} - train_loss={train_loss:.4f} val_loss={val_loss:.4f} val_acc={val_acc:.4f}')

    train_time = time.perf_counter() - start
    gpu_mem_mb = torch.cuda.max_memory_allocated() / (1024**2) if device.type == 'cuda' else 0.0
    return history, train_time, gpu_mem_mb

history, train_time_sec, gpu_memory_mb = train_model(model, train_loader, test_loader, EPOCHS)

Epoch 1/5 - train_loss=2.0409 val_loss=1.8527 val_acc=0.3102
Epoch 2/5 - train_loss=1.7961 val_loss=1.7156 val_acc=0.3891
Epoch 3/5 - train_loss=1.6863 val_loss=1.6427 val_acc=0.4141
Epoch 4/5 - train_loss=1.6263 val_loss=1.5983 val_acc=0.4268
Epoch 5/5 - train_loss=1.5902 val_loss=1.5736 val_acc=0.4379


In [ ]:
def predict(model, loader):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            logits = model(x)
            pred = logits.argmax(dim=1).cpu().numpy()
            y_pred.extend(pred.tolist())
            y_true.extend(y.numpy().tolist())
    return np.array(y_true), np.array(y_pred)

def inference_time_per_sample(model, loader):
    model.eval()
    t0 = time.perf_counter()
    total = 0
    with torch.no_grad():
        for x, _ in loader:
            x = x.to(device)
            _ = model(x)
            total += x.size(0)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    dt = time.perf_counter() - t0
    return dt / max(total, 1)

y_true, y_pred = predict(model, test_loader)
acc = accuracy_score(y_true, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)
metrics_df = pd.DataFrame([{
    'model': 'HQViT',
    'accuracy': acc,
    'precision': precision,
    'recall': recall,
    'f1': f1,
    'training_time_sec': train_time_sec,
    'gpu_memory_mb': gpu_memory_mb,
    'num_parameters': sum(p.numel() for p in model.parameters() if p.requires_grad),
    'inference_time_per_sample_sec': inference_time_per_sample(model, test_loader),
}])
display(metrics_df)

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('HQViT Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Val Loss')
plt.plot(history['val_acc'], label='Val Accuracy')
plt.title('HQViT Training Curves')
plt.xlabel('Epoch')
plt.legend()
plt.grid(alpha=0.3)
plt.show()